# 2.4 Edge Deployment Feasibility

Benchmark the LightGBM baseline model for edge deployment: measure inference latency, memory footprint, and throughput in Python (native LightGBM + ONNX Runtime), then export ONNX for Rust/WASM benchmarking.

**Model benchmarked:**
- Full LGBM (57 features, 286 trees, max_depth=6, num_leaves=68)

**Target constraints:** <5ms inference, 50k req/s, WASM runtime, no GPU.

**Note:** Pruned model (32 features, 99% threshold) is NOT used for deployment — pruning causes PR-AUC to drop from 1.0 to 0.879 and credential_stuffing recall to collapse to 0.0. The full 57-feature model is the deployment candidate.

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import time
import json
import tracemalloc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import onnxruntime as ort

from src.label_joining import join_labels
from src.features import compute_per_request_features, compute_session_features
from src.pipeline import compute_sample_weights
from src.model import (
    train_model, get_feature_columns, temporal_train_test_split,
    get_feature_importance, prune_features,
)
from src.export import export_to_onnx, validate_onnx_export

In [2]:
requests = pd.read_csv("data/http_requests.csv", parse_dates=["timestamp"])
headers = pd.read_csv("data/request_headers.csv")
labels = pd.read_csv("data/incident_labels.csv", parse_dates=["active_from", "active_until", "labeled_at"])

df = join_labels(requests, labels)
df = compute_per_request_features(df, headers)
df = compute_session_features(df)
df["sample_weight"] = compute_sample_weights(df)

train_df, test_df = temporal_train_test_split(df, test_date="2025-01-10")
feature_cols = get_feature_columns(df)

X_train = train_df[feature_cols].astype(float)
y_train = train_df["is_malicious"].astype(int)
w_train = train_df["sample_weight"]
X_test = test_df[feature_cols].astype(float)

# Best LGBM params from 2.3 Optuna tuning
lgbm_params = {
    "n_estimators": 286, "num_leaves": 68, "max_depth": 6,
    "learning_rate": 0.0981, "subsample": 0.6091,
    "colsample_bytree": 0.5194, "min_child_samples": 49,
    "scale_pos_weight": 78.91, "reg_alpha": 3.3316,
    "reg_lambda": 0.0442, "verbose": -1, "metric": "average_precision",
}
model_full = train_model("lgbm", lgbm_params, X_train, y_train, w_train)

print(f"Full model: {len(feature_cols)} features, {lgbm_params['n_estimators']} trees, max_depth={lgbm_params['max_depth']}")

Full model: 57 features, 286 trees, max_depth=6


## 1. ONNX Export & Validation

Export both models to ONNX format and validate predictions match the originals.

In [3]:
os.makedirs("edge-inference/model", exist_ok=True)

onnx_full_path = "edge-inference/model/model_full.onnx"

export_to_onnx(model_full, feature_cols, onnx_full_path)

val_full = validate_onnx_export(model_full, onnx_full_path, X_test)

print("ONNX Validation:")
print(f"  Full model  — max abs error: {val_full['max_abs_error']:.2e}, match: {val_full['match']}")
print(f"\nFile size:")
print(f"  Full ONNX:   {os.path.getsize(onnx_full_path) / 1024:.1f} KB")

ONNX Validation:
  Full model  — max abs error: 9.08e-08, match: True

File size:
  Full ONNX:   38.0 KB


2026-07-17 15:18:34.805 Python[71751:1011605] 2026-07-17 15:18:34.798941 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {21143} for output label


In [4]:
sample_row = X_test[feature_cols].iloc[0].astype(float).to_dict()
sample_input = {
    "feature_names": feature_cols,
    "values": [float(sample_row[f]) for f in feature_cols],
}
with open("edge-inference/model/sample_input.json", "w") as f:
    json.dump(sample_input, f, indent=2)
print(f"Sample input saved: {len(feature_cols)} features")
print(f"Features: {feature_cols}")

Sample input saved: 57 features
Features: ['path_depth', 'path_length', 'path_entropy', 'path_has_params', 'ua_length', 'ua_entropy', 'ua_is_browser', 'ua_is_bot_library', 'hour_of_day', 'is_sensitive_endpoint', 'header_count', 'has_accept_language', 'has_referer', 'has_cookie', 'has_authorization', 'ip_1m_request_count', 'ip_1m_unique_paths', 'ip_1m_path_entropy', 'ip_1m_method_diversity', 'ip_1m_sensitive_endpoint_ratio', 'ip_1m_inter_request_time_mean', 'ip_1m_inter_request_time_std', 'ip_5m_request_count', 'ip_5m_unique_paths', 'ip_5m_path_entropy', 'ip_5m_method_diversity', 'ip_5m_sensitive_endpoint_ratio', 'ip_5m_inter_request_time_mean', 'ip_5m_inter_request_time_std', 'ip_30m_request_count', 'ip_30m_unique_paths', 'ip_30m_path_entropy', 'ip_30m_method_diversity', 'ip_30m_sensitive_endpoint_ratio', 'ip_30m_inter_request_time_mean', 'ip_30m_inter_request_time_std', 'tls_1m_request_count', 'tls_1m_unique_paths', 'tls_1m_path_entropy', 'tls_1m_method_diversity', 'tls_1m_sensitive_e

## 2. Inference Latency Benchmarks

Measure per-request latency for single and batch inference across two configurations:
- LightGBM native Python (57 features)
- ONNX Runtime Python (57 features)

Each benchmark runs 1000 iterations for single-request, reporting p50/p95/p99.

In [5]:
def benchmark_latency(predict_fn, X_input, n_iterations=1000):
    """Run predict_fn n_iterations times, return latency stats in ms."""
    # Warmup
    for _ in range(10):
        predict_fn(X_input)

    latencies = []
    for _ in range(n_iterations):
        start = time.perf_counter()
        predict_fn(X_input)
        elapsed = (time.perf_counter() - start) * 1000  # ms
        latencies.append(elapsed)

    latencies = np.array(latencies)
    return {
        "mean_ms": float(np.mean(latencies)),
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "min_ms": float(np.min(latencies)),
        "max_ms": float(np.max(latencies)),
    }

In [6]:
# Prepare inputs
single_full = X_test[feature_cols].iloc[:1].astype(float)

# ONNX session
sess_full = ort.InferenceSession(onnx_full_path)
input_name_full = sess_full.get_inputs()[0].name

results = {}
batch_sizes = [1, 10, 100, 1000]

for batch_size in batch_sizes:
    batch_full = X_test[feature_cols].iloc[:batch_size].astype(float)

    n_iter = 1000 if batch_size <= 100 else 100

    def lgbm_full_fn(x, _m=model_full): return _m.predict_proba(x)
    def onnx_full_fn(x, _s=sess_full, _n=input_name_full):
        return _s.run(None, {_n: x.values.astype(np.float32)})

    results[f"lgbm_full_batch{batch_size}"] = benchmark_latency(lgbm_full_fn, batch_full, n_iter)
    results[f"onnx_full_batch{batch_size}"] = benchmark_latency(onnx_full_fn, batch_full, n_iter)
    print(f"Batch size {batch_size} done.")

Batch size 1 done.


2026-07-17 15:18:36.053 Python[71751:1011605] 2026-07-17 15:18:36.053462 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-17 15:18:36.053 Python[71751:1011605] 2026-07-17 15:18:36.053679 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-17 15:18:36.053 Python[71751:1011605] 2026-07-17 15:18:36.053895 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-17 15:18:36.053 Python[71751:1011605] 2026-07-17 15:18:36.053990 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-17 15:18:36.054 Python[71751:1011605] 2026-07-17 15:18:36.054072 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSiz

Batch size 10 done.


2026-07-17 15:18:36.755 Python[71751:1011605] 2026-07-17 15:18:36.755303 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-17 15:18:36.755 Python[71751:1011605] 2026-07-17 15:18:36.755612 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-17 15:18:36.756 Python[71751:1011605] 2026-07-17 15:18:36.756013 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-17 15:18:36.756 Python[71751:1011605] 2026-07-17 15:18:36.756230 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-17 15:18:36.756 Python[71751:1011605] 2026-07-17 15:18:36.756411 [W:onnxruntime:, execution_frame.cc:913 VerifyOutpu

Batch size 100 done.


Batch size 1000 done.


2026-07-17 15:18:37.028 Python[71751:1011605] 2026-07-17 15:18:37.028759 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-07-17 15:18:37.029 Python[71751:1011605] 2026-07-17 15:18:37.029835 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-07-17 15:18:37.030 Python[71751:1011605] 2026-07-17 15:18:37.030809 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-07-17 15:18:37.031 Python[71751:1011605] 2026-07-17 15:18:37.031716 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {1000} for output label
2026-07-17 15:18:37.032 Python[71751:1011605] 2026-07-17 15:18:37.032585 [W:onnxruntime:, execution_frame.cc:913 VerifyO

In [7]:
rows = []
for batch_size in batch_sizes:
    for runtime in ["lgbm", "onnx"]:
        key = f"{runtime}_full_batch{batch_size}"
        r = results[key]
        rows.append({
            "Runtime": f"{runtime.upper()} (57f)",
            "Batch": batch_size,
            "p50 (ms)": f"{r['p50_ms']:.3f}",
            "p95 (ms)": f"{r['p95_ms']:.3f}",
            "p99 (ms)": f"{r['p99_ms']:.3f}",
            "mean (ms)": f"{r['mean_ms']:.3f}",
        })

latency_df = pd.DataFrame(rows)
print("Inference Latency (lower is better):")
print(latency_df.to_string(index=False))

Inference Latency (lower is better):
   Runtime  Batch p50 (ms) p95 (ms) p99 (ms) mean (ms)
LGBM (57f)      1    0.557    0.635    0.766     0.570
ONNX (57f)      1    0.020    0.022    0.025     0.020
LGBM (57f)     10    0.577    0.624    0.785     0.612
ONNX (57f)     10    0.053    0.066    0.086     0.055
LGBM (57f)    100    0.630    0.678    0.754     0.637
ONNX (57f)    100    0.144    0.174    0.221     0.146
LGBM (57f)   1000    1.040    1.145    1.284     1.061
ONNX (57f)   1000    0.792    0.932    0.975     0.809


## 3. Memory Footprint

Compare model sizes on disk (serialized) and peak runtime memory during inference.

In [8]:
import tempfile

# Disk size — joblib pkl
with tempfile.NamedTemporaryFile(suffix=".pkl") as f:
    joblib.dump(model_full, f.name)
    pkl_full_size = os.path.getsize(f.name)

onnx_full_size = os.path.getsize(onnx_full_path)

# Runtime memory — tracemalloc during inference
def measure_runtime_memory(predict_fn, X_input, n_runs=100):
    tracemalloc.start()
    for _ in range(n_runs):
        predict_fn(X_input)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

mem_lgbm_full = measure_runtime_memory(
    lambda x: model_full.predict_proba(x), single_full,
)
mem_onnx_full = measure_runtime_memory(
    lambda x: sess_full.run(None, {input_name_full: x.values.astype(np.float32)}),
    single_full,
)

print("Memory Footprint:")
print(f"\n  Disk (serialized):")
print(f"    Full  — PKL: {pkl_full_size/1024:.1f} KB, ONNX: {onnx_full_size/1024:.1f} KB")
print(f"\n  Runtime peak memory (100 inferences):")
print(f"    LGBM full:   {mem_lgbm_full/1024:.1f} KB")
print(f"    ONNX full:   {mem_onnx_full/1024:.1f} KB")

Memory Footprint:



  Disk (serialized):
    Full  — PKL: 95.5 KB, ONNX: 38.0 KB

  Runtime peak memory (100 inferences):
    LGBM full:   138.3 KB
    ONNX full:   10.8 KB


## 4. Throughput Feasibility

Calculate how many CPU cores are needed to sustain 50k req/s based on measured single-request p99 latency.

Formula: `threads_needed = target_rps × p99_latency_seconds`

In [9]:
target_rps = 50_000

print("Throughput Feasibility (target: 50,000 req/s):")
print(f"\n{'Runtime':<25} {'p99 (ms)':>10} {'Throughput/thread':>20} {'Threads for 50k':>18}")
print("-" * 75)

for runtime in ["lgbm", "onnx"]:
    key = f"{runtime}_full_batch1"
    p99 = results[key]["p99_ms"]
    throughput_per_thread = 1000.0 / p99
    threads_needed = target_rps * (p99 / 1000.0)
    label = f"{runtime.upper()} (57f)"
    print(f"{label:<25} {p99:>10.3f} {throughput_per_thread:>17.0f}/s {threads_needed:>17.1f}")

print(f"\nA typical edge node has 4-8 CPU cores.")
print(f"If threads_needed <= 8, the model fits within a single edge node's capacity.")

Throughput Feasibility (target: 50,000 req/s):

Runtime                     p99 (ms)    Throughput/thread    Threads for 50k
---------------------------------------------------------------------------
LGBM (57f)                     0.766              1306/s              38.3
ONNX (57f)                     0.025             40814/s               1.2

A typical edge node has 4-8 CPU cores.
If threads_needed <= 8, the model fits within a single edge node's capacity.


## 5. Summary

Key findings from Python benchmarks. The Rust/WASM benchmarks in `edge-inference/` provide the production-representative numbers.

In [10]:
onnx_full_single = results["onnx_full_batch1"]

print("=" * 60)
print("EDGE DEPLOYMENT FEASIBILITY — PYTHON BENCHMARKS")
print("=" * 60)
print(f"\nDeployment candidate: ONNX Full (57 features)")
print(f"  Single-request p99: {onnx_full_single['p99_ms']:.3f} ms")
print(f"  {'✓ PASS' if onnx_full_single['p99_ms'] < 5.0 else '✗ FAIL'}: under 5ms latency budget")
p99_s = onnx_full_single["p99_ms"] / 1000
threads = target_rps * p99_s
print(f"  Threads for 50k req/s: {threads:.1f}")
print(f"  {'✓ PASS' if threads <= 8 else '✗ FAIL'}: fits in ≤8 cores")
print(f"\n  ONNX file: {onnx_full_size/1024:.1f} KB")
print(f"  ONNX validation: max_abs_error = {val_full['max_abs_error']:.2e}")
print(f"\nNext: run Rust/WASM benchmarks in edge-inference/")

EDGE DEPLOYMENT FEASIBILITY — PYTHON BENCHMARKS

Deployment candidate: ONNX Full (57 features)
  Single-request p99: 0.025 ms
  ✓ PASS: under 5ms latency budget
  Threads for 50k req/s: 1.2
  ✓ PASS: fits in ≤8 cores

  ONNX file: 38.0 KB
  ONNX validation: max_abs_error = 9.08e-08

Next: run Rust/WASM benchmarks in edge-inference/
